# VisionSetil Kaggle Benchmark

This notebook orchestrates the batch evaluation of the **VisionSetil** mushroom identification pipeline on Kaggle. It enables running deep learning benchmarks over large image datasets, persistent saving of cropped regions and embeddings, and generating taxonomic and safety reports.

## 1. Environment Check
First, check if a GPU is available and inspect system resources.

In [ ]:
!nvidia-smi
import torch
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))

## 2. Install Dependencies
Install the required libraries for running VisionSetil and processing evaluation metrics.

In [ ]:
# Clone the repository or navigate to the project directory if uploaded
!git clone https://github.com/AlonsoAlviraa/VisionSetil.git
%cd VisionSetil

!pip install -r backend/requirements.txt
!pip install psutil gputil

## 3. Paths and Config
Inspect or edit the runtime configurations. Make sure to link your Kaggle Datasets containing images and labels.

In [ ]:
import json
from pathlib import Path

# Load example config
config_file = "kaggle/kaggle_run_config.example.json"
with open(config_file, "r") as f:
    config = json.load(f)

print(json.dumps(config, indent=2))

## 4. Model Status
Query the model registry status to verify if real deep learning backends or fallbacks (mocks) are loaded.

In [ ]:
import sys
sys.path.insert(0, str(Path("backend").resolve()))
from fastapi.testclient import TestClient
from app.main import app

client = TestClient(app)
status_response = client.get("/models/status")
print(json.dumps(status_response.json(), indent=2))

## 5. Dataset Preview
Verify the labels JSON path and load a preview of the dataset.

In [ ]:
dataset_path = config.get("dataset_path")
if Path(dataset_path).exists():
    with open(dataset_path, "r") as f:
        preview = json.load(f)
    print(f"Total cases in dataset: {len(preview)}")
    print("Preview of first item:")
    print(json.dumps(preview[0], indent=2))
else:
    print(f"Dataset labels not found at {dataset_path}. Using fallback mock dataset validation instead.")

## 6. Run Benchmark
Execute the Kaggle benchmark orchestrator script. Customize case limits or filters if needed.

In [ ]:
# Executing the orchestrator script
!python kaggle/run_kaggle_benchmark.py \
  --config kaggle/kaggle_run_config.example.json \
  --max-cases 100 \
  --shuffle \
  --seed 42

## 7. Summarize Results
Format and display the summary from the generated report JSON.

In [ ]:
!python eval/scripts/summarize_eval.py --report /kaggle/working/visionsetil_outputs/real_report.json

## 8. Show Key Metrics
Read the generated structured reports and output primary biological and safety metrics.

In [ ]:
import pandas as pd

report_file = "/kaggle/working/visionsetil_outputs/real_report.json"
if Path(report_file).exists():
    with open(report_file, "r") as f:
        metrics_data = json.load(f)
    
    m = metrics_data.get("metrics", {})
    key_metrics = {
        "Species Top-1 Accuracy": f"{m.get('species_top1_accuracy', 0)*100:.2f}%",
        "Species Top-5 Accuracy": f"{m.get('species_top5_accuracy', 0)*100:.2f}%",
        "Genus Accuracy": f"{m.get('genus_accuracy', 0)*100:.2f}%",
        "False Safe Rate": f"{m.get('false_safe_rate', 0)*100:.2f}% (Must be 0%)",
        "Toxic Not Flagged Rate": f"{m.get('toxic_not_flagged_rate', 0)*100:.2f}%",
        "Dangerous Bypass Rate": f"{m.get('dangerous_case_without_human_review_rate', 0)*100:.2f}%",
        "Overconfident Wrong Rate": f"{m.get('overconfident_wrong_rate', 0)*100:.2f}%",
        "Production Readiness": metrics_data.get("production_readiness", {}).get("status")
    }
    df = pd.DataFrame(list(key_metrics.items()), columns=["Metric", "Value"])
    display(df)
else:
    print("Report file not found. Ensure the benchmark completed successfully.")

## 9. Inspect Dangerous Failures
Examine observations belonging to high-risk genera or toxic risk levels that did not pass safety gating.

In [ ]:
failures_file = "/kaggle/working/visionsetil_outputs/dangerous_failures.json"
if Path(failures_file).exists():
    with open(failures_file, "r") as f:
        dangerous_fails = json.load(f)
    print(f"Found {len(dangerous_fails)} dangerous classification failures:")
    if dangerous_fails:
        df_fails = pd.DataFrame(dangerous_fails)
        display(df_fails.head(10))
else:
    print("No dangerous failures file found.")

## 10. Download Outputs
Locate and verify all final output reports ready for download.

In [ ]:
print("Artifacts in output folder /kaggle/working/visionsetil_outputs/:")
output_path = Path("/kaggle/working/visionsetil_outputs")
if output_path.exists():
    for file in output_path.glob("*"):
        print(f"  - {file.name} ({file.stat().st_size} bytes)")
else:
    print("Output directory does not exist yet.")